In [1]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/901947899065852447'), creation_time=1778952120616, experiment_id='901947899065852447', last_update_time=1778952120616, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [1]:
# =========================================================
# IMPORTS
# =========================================================
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd     
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE


In [3]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
...,...,...
36788,jesus,0
36789,kya bhai pure saal chutiya banaya modi aur jab...,1
36790,downvote karna tha par upvote hogaya,0
36791,haha nice,1


In [5]:
# =========================================================
# IMPORTS
# =========================================================
import optuna
import mlflow
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# =========================================================
# REMOVE NaN TARGETS
# =========================================================
df = df.dropna(
    subset=['category']
)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df['clean_comment'],
    df['category'],

    test_size=0.2,

    random_state=42,

    stratify=df['category']
)

# =========================================================
# FIXED TF-IDF SETTINGS
# =========================================================
ngram_range = (1, 3)

max_features = 3000## slow learns fails for 5000

# =========================================================
# CROSS VALIDATION
# =========================================================
cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

# =========================================================
# OPTUNA OBJECTIVE
# =========================================================
def objective_knn(trial):

    n_neighbors = trial.suggest_int(
        'n_neighbors',
        3,
        25
    )

    weights = trial.suggest_categorical(
        'weights',
        ['uniform', 'distance']
    )

    # -----------------------------------------------------
    # PIPELINE
    # -----------------------------------------------------
    pipeline = Pipeline([

        (
            "tfidf",

            TfidfVectorizer(

                ngram_range=ngram_range,

                max_features=max_features,

                min_df=2,

                max_df=0.95
            )
        ),

        (
            "model",

            KNeighborsClassifier(

                n_neighbors=n_neighbors,

                weights=weights,

                metric='cosine',

                n_jobs=-1
            )
        )
    ])

    # -----------------------------------------------------
    # CV SCORE
    # -----------------------------------------------------
    scores = cross_val_score(

        pipeline,

        X_train_text,

        y_train,

        cv=cv,

        scoring='f1_macro',

        n_jobs=-1
    )

    return np.mean(scores)

# =========================================================
# OPTUNA STUDY
# =========================================================
study = optuna.create_study(
    direction="maximize"
)

study.optimize(

    objective_knn,

    n_trials=20
)

# =========================================================
# BEST PARAMETERS
# =========================================================
print("=" * 60)
print("BEST PARAMETERS")
print(study.best_params)
print("=" * 60)

# =========================================================
# FINAL PIPELINE
# =========================================================
best_pipeline = Pipeline([

    (
        "tfidf",

        TfidfVectorizer(

            ngram_range=ngram_range,

            max_features=max_features,

            min_df=2,

            max_df=0.95
        )
    ),

    (
        "model",

        KNeighborsClassifier(

            n_neighbors=study.best_params['n_neighbors'],

            weights=study.best_params['weights'],

            metric='cosine',

            n_jobs=-1
        )
    )
])

# =========================================================
# TRAIN FINAL MODEL
# =========================================================
best_pipeline.fit(

    X_train_text,

    y_train
)

# =========================================================
# PREDICTIONS
# =========================================================
y_pred = best_pipeline.predict(
    X_test_text
)

# =========================================================
# METRICS
# =========================================================
accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(

    y_test,

    y_pred,

    average='macro'
)

weighted_f1 = f1_score(

    y_test,

    y_pred,

    average='weighted'
)

# =========================================================
# RESULTS
# =========================================================
print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# MLFLOW
# =========================================================
with mlflow.start_run():

    mlflow.set_tag(
        "model",
        "KNN"
    )

    mlflow.log_param(
        "ngram_range",
        ngram_range
    )

    mlflow.log_param(
        "max_features",
        max_features
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "macro_f1",
        macro_f1
    )

    mlflow.log_metric(
        "weighted_f1",
        weighted_f1
    )

    mlflow.sklearn.log_model(
        best_pipeline,
        "knn_pipeline"
    )

[I 2026-05-17 14:44:35,948] A new study created in memory with name: no-name-a7b4770c-afcd-41c9-80a2-22c32350ed02
[I 2026-05-17 14:45:22,379] Trial 0 finished with value: 0.4833746122361691 and parameters: {'n_neighbors': 19, 'weights': 'uniform'}. Best is trial 0 with value: 0.4833746122361691.
[I 2026-05-17 14:45:53,886] Trial 1 finished with value: 0.485426581770542 and parameters: {'n_neighbors': 16, 'weights': 'distance'}. Best is trial 1 with value: 0.485426581770542.
[I 2026-05-17 14:46:18,940] Trial 2 finished with value: 0.47152747049077137 and parameters: {'n_neighbors': 10, 'weights': 'uniform'}. Best is trial 1 with value: 0.485426581770542.
[W 2026-05-17 14:46:31,889] Trial 3 failed with parameters: {'n_neighbors': 17, 'weights': 'distance'} because of the following error: The value nan is not acceptable.
[W 2026-05-17 14:46:31,897] Trial 3 failed with value np.float64(nan).
[I 2026-05-17 14:46:45,926] Trial 4 finished with value: 0.47378187561730883 and parameters: {'n_ne

BEST PARAMETERS
{'n_neighbors': 25, 'weights': 'distance'}
Accuracy    : 0.5268
Macro F1    : 0.4937
Weighted F1 : 0.5100

Classification Report:

              precision    recall  f1-score   support

          -1       0.75      0.26      0.38      1650
           0       0.44      0.82      0.57      2529
           1       0.68      0.43      0.53      3154

    accuracy                           0.53      7333
   macro avg       0.62      0.50      0.49      7333
weighted avg       0.61      0.53      0.51      7333



2026/05/17 14:52:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 14:52:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
